In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df =  pd.read_csv('datasets-day-2/dataset-1.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
%pip install missingno

In [ ]:
import missingno as msn
msn.matrix(df)

In [ ]:
df.isnull().sum()

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
for i in df.columns:
    
    if i in numerical_cols:
        plt.figure(figsize=(4,4))
        sns.boxplot(df[i])
        plt.show()

In [ ]:
df['x8'].nunique()

In [ ]:
'''for i in df['x8']:
    df[i]=i.lower().strip().replace("@","")
df['x8']'''

In [ ]:
df['x8'].head()

In [ ]:
df.info()

X3 has ? --> replace to nan --> type to float

In [ ]:
df['x3'] = df['x3'].replace('?', np.nan).astype('float64')

In [ ]:
df['x3'].isnull().sum()

In [ ]:
q1 = df['x3'].quantile(0.25)
q3 = df['x3'].quantile(0.75)

iqr = q3-q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df['x3']< lower_bound ) | (df['x3']> upper_bound)]

outliers

FIll null Values

In [ ]:
from sklearn.impute import SimpleImputer

impute = SimpleImputer(strategy='median')
df['x3']=impute.fit_transform(df[['x3']])

In [ ]:
df['x3'].isnull().sum()

In [ ]:
df['x3'].dtype

Heatmap

In [ ]:
sns.heatmap(df.select_dtypes(include=['int64','float64']).corr(),annot=True)

In [ ]:
df.info()

In [ ]:
X=df.drop('target',axis=1)
y=df['target']


In [ ]:
num_col = X.select_dtypes(include=['int64','float64'])
cat_col =X.select_dtypes(include=['object'])


In [ ]:
cat_col

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
X['x8_enc'] = le.fit_transform(X['x8'])

X.head(15)


In [ ]:

with open('encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
std = StandardScaler()

x_train_std = std.fit_transform(x_train)
x_test = std.transform(x_test)

In [ ]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(std, f)

<h1>Linear reg</h1>

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(x_train_std,y_train)

In [ ]:
y_pred = lr.predict(x_test)


In [ ]:
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_error

print("mean sq error :", mean_squared_error(y_test,y_pred))

print("r2_score :", r2_score(y_test,y_pred))

print("root_mean_squared_error :", root_mean_squared_error(y_test,y_pred))


<H1>poly</h1>

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

ply= PolynomialFeatures(degree=2)
x_train_std_poly = ply.fit_transform(x_train_std)
x_test_poly = ply.transform(x_test)



In [ ]:
model = LinearRegression()
model.fit(x_train_std_poly, y_train)
y_pred_poly = model.predict(x_test_poly)

print("Mean Squared Error:", mean_squared_error(y_test, y_pred_poly))
print("R2 Score:", r2_score(y_test, y_pred_poly))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred_poly))

<h1>SVR</h1>

In [ ]:
from sklearn.svm import SVR

svm = SVR(kernel='rbf',C=100,epsilon=0.1,gamma=0.1)
svm.fit(x_train_std,y_train)


In [ ]:
from sklearn.model_selection import GridSearchCV
svr = SVR()
svr_params = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'epsilon': [0.01, 0.1, 0.2, 0.5]
}
svr_grid = GridSearchCV(
    estimator=svr,
    param_grid=svr_params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)
svr_grid.fit(x_train_std, y_train)
print("=" * 50)
print("SVR - Best Parameters:", svr_grid.best_params_)
print("SVR - Best Score:", -svr_grid.best_score_)
print("=" * 50)


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error,mean_absolute_error   
y_pred_svr = svm.predict(x_test)

print("Mean Squared Error:", mean_squared_error(y_test, y_pred_svr))
print("R2 Score:", r2_score(y_test, y_pred_svr))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred_svr))
print("Root Mean absolute Error:", mean_absolute_error(y_test, y_pred_svr))

SVM is the High Accuracy model

In [ ]:
import pickle

with open('svm_model.pkl', 'wb') as file:
    pickle.dump(svm, file)
print("SVM model saved to svm_model.pkl")

<h1>Dt</h1>

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dtr = DecisionTreeRegressor(criterion='squared_error',max_depth=None,max_features='log2',min_samples_leaf=4,min_samples_split=2,random_state=42,splitter='best')
dtr.fit(x_train_std,y_train)

In [ ]:
dtr = DecisionTreeRegressor(random_state=42)
dtr_params = {
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['auto', 'sqrt', 'log2', None],
    'splitter': ['best', 'random']
}
dtr_grid = GridSearchCV(
    estimator=dtr,
    param_grid=dtr_params,
    cv=2,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)
dtr_grid.fit(x_train_std, y_train)
print("=" * 50)
print("DTR - Best Parameters:", dtr_grid.best_params_)
print("DTR - Best Score:", -dtr_grid.best_score_)
print("=" * 50)

In [ ]:
y_pred_dt = dtr.predict(x_test)

print("Mean Squared Error:", mean_squared_error(y_test, y_pred_dt))
print("R2 Score:", r2_score(y_test, y_pred_dt))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred_dt))

<h1>Random forest</h1>

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rfr = RandomForestRegressor(criterion='squared_error',n_estimators=100)
rfr.fit(x_train_std,y_train)

In [ ]:
y_pred_rf = rfr.predict(x_test)

print("Mean Squared Error:", mean_squared_error(y_test, y_pred_rf))
print("R2 Score:", r2_score(y_test, y_pred_rf))
print("Root Mean Squared Error:", root_mean_squared_error(y_test, y_pred_rf))

In [ ]:

# Load all three pickle files
with open('svm_model.pkl', 'rb') as f:
    svm_model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
# Raw input data
x1, x2, x3, x4, x5, x6, x7 = 4, 97, 46, 1835, 20.5, 70, 2
x8 = "volkswagen 1131 deluxe sedan"
x8_encoded = label_encoder.transform([x8])[0]

input = np.array([[x1, x2, x3, x4, x5, x6, x7, x8_encoded]])
i_s = std.transform(input)

print("SVM:", svm.predict(i_s))
print("RFR :", rfr.predict(i_s))
print("DT :", dtr.predict(i_s))
print("l :", lr.predict(i_s))


